# Video Engagement Analytics

Executed walkthrough of the actual aggregate outputs. Run the scripts in the README to regenerate the underlying SQL marts. No user-level records are embedded.

In [1]:
from pathlib import Path
import json, pandas as pd
ROOT = Path.cwd() if (Path.cwd()/'reports/results.json').exists() else Path.cwd().parent
result = json.loads((ROOT/'reports/results.json').read_text())
print(json.dumps(result['summary'], indent=2))

{
  "exposures": 10683905,
  "users": 1000,
  "watch_hours": 46384.58891527428,
  "capped_watch_hours": 42216.72770387216,
  "seconds_per_exposure": 15.629539957065083,
  "long_view_rate": 0.27407806415350944,
  "completion_rate": 0.16531726929432639
}


## Quality before interpretation
Exact duplicates are removed. Zero-duration exposures cannot support completion metrics; sensitivity analyses quantify their watch-time effect. Upstream date and feedback discrepancies are retained as caveats.

In [2]:
q=result['quality']
print(json.dumps({k:q[k] for k in ['counts','exact_duplicate_rows_removed','zero_duration_rows','long_view_definition_mismatches','utc_plus_8_date_mismatches']},indent=2))

{
  "counts": {
    "raw": 11756073,
    "deduplicated": 11661848,
    "classified": 11661848,
    "events": 10725622,
    "standard_events": 10683905,
    "user_day": 27955
  },
  "exact_duplicate_rows_removed": 94225,
  "zero_duration_rows": 936226,
  "long_view_definition_mismatches": 55975,
  "utc_plus_8_date_mismatches": 75566
}


## Watch-hour decomposition
The three factors use the same weekly population. Contributions are arithmetic, not causal.

In [3]:
print(pd.read_csv(ROOT/'reports/weekly_comparison.csv').to_string(index=False))
print(result['shapley_hours'])
assert abs(sum(result['shapley_hours'].values())-result['weekly_watch_change_hours'])<1e-6

    period start_date   end_date  observed_days  exposures  active_users  exposures_per_user  seconds_per_exposure  watch_hours  capped_watch_hours
  baseline 2022-04-08 2022-04-14              7    2318265           957         2422.429467             15.854470 10209.684396         9324.680122
comparison 2022-04-29 2022-05-05              7    2653922           996         2664.580321             15.365718 11327.615897        10312.390889
{'active_users': 430.0236247312264, 'exposures_per_user': 1025.2175180708716, 'seconds_per_exposure': -337.3096414135296}


## Return and audience behavior
D7 has a user-level eligible denominator and an exact-day outcome. Prior-week quartiles use no future data.

In [4]:
print(pd.read_csv(ROOT/'reports/pooled_return.csv').to_string(index=False))
print(pd.read_csv(ROOT/'reports/audience_segments.csv').to_string(index=False))

 horizon  eligible_users  returned_users  return_rate  wilson_lower  wilson_upper
       1            1000             954     0.954000      0.939188      0.965338
       7             995             929     0.933668      0.916477      0.947524
 engagement_quartile  users  min_early_minutes  max_early_minutes  avg_early_minutes  avg_next_active_days  avg_next_minutes  returned_users  next_week_return_rate
                   1    240           0.000000         269.529733         124.407371              5.629167        212.418475           236.0               0.983333
                   2    239         269.539533         510.021067         390.641286              6.435146        435.828715           237.0               0.991632
                   3    239         510.353767         862.654233         675.129265              6.615063        643.305966           239.0               1.000000
                   4    239         862.700217        3566.745833        1372.402230              

## Sensitivity and checks

In [5]:
print(pd.read_csv(ROOT/'reports/watch_sensitivity.csv').to_string(index=False))
assert all(result['checks'].values())
print(f"{len(result['checks'])} reconciliation checks passed")

                definition  baseline_hours  comparison_hours  change_pct
                 raw_watch    10209.684396      11327.615897   10.949717
           duration_capped     9324.680122      10312.390889   10.592436
including_unknown_duration    10544.450983      11762.541684   11.551959
12 reconciliation checks passed
